# Establish connection to pawsey0411

Startup version validation. Catch dependency issues fail before scanning data

In [ ]:
import sys,json
from importlib import import_module
from importlib.metadata import version, PackageNotFoundError

REQUIRED_EXACT = {
    "fsspec": "2026.3.0",
    "s3fs": "2026.3.0",
    "kerchunk": "0.2.10",
    "virtualizarr": "2.6.0",
}
errors = []
report = {}

if sys.version_info <(3,12):
    errors.append("Python 3.12+ required for virtualizarr==2.6.0. Check environment configuration.")

for pkg, expected_version in REQUIRED_EXACT.items():
    try:
        got_version= version(pkg)
        report[pkg]= got_version
        if got_version != expected_version:
            errors.append(f"{pkg} Version expected: {expected_version}, got: {got_version}")
    except PackageNotFoundError:
        errors.append(f"{pkg} missing")

for mod in ["fsspec", "fastparquet","s3fs", "xarray", "zarr", "kerchunk", "virtualizarr"]:
    try:
        import_module(mod)
    except Exception as exc:
        errors.append(f"import {mod} failed: {type(exc).__name__}: {exc}")

# s3fs version == fsspec version == 2026.3.0
if report.get("fsspec") and report.get("s3fs") and report["fsspec"] != report["s3fs"]:
    errors.append(f"fsspec/s3fs mismatch: {report['fsspec']} vs {report['s3fs']}")

if errors:
    raise RuntimeError("Runtime readiness FAILED:\n- " + "\n- ".join(errors))

print("Runtime readiness PASSED")
print(json.dumps(report, indent=2, sort_keys=True))

## Load YAML config
Establish ECMWF, DPIRD object path

In [ ]:
import re, os, yaml

"""Load in yaml config"""
def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

cfg = load_yaml("../configs/config.yaml")
kp = cfg.get("kerchunk_pipeline", {})

#Validate any missing yaml entries
required_top = ["s3", "source_flows", "output", "execution"]
missing_top = [k for k in required_top if k not in kp]
if missing_top:
    raise ValueError(f"Missing kerchunk_pipeline sections: {missing_top}")

s3 = kp["s3"]
for k in ["endpoint_url", "bucket", "secret_scope", "access_key_secret_name", "secret_key_secret_name"]:
    if not s3.get(k):
        raise ValueError(f"Missing s3.{k}")

flows = kp["source_flows"]
if len(flows) != 2:
    raise ValueError("Expected exactly 2 source_flows for phase 2")

flow_ids = {f.get("id") for f in flows}
if flow_ids != {"ecmwf_daily_nc", "dpird_final_singleton"}:
    raise ValueError(f"Unexpected flow ids: {flow_ids}")

ecmwf = [f for f in flows if f["id"] == "ecmwf_daily_nc"][0]
dpird = [f for f in flows if f["id"] == "dpird_final_singleton"][0]

"""Check matching types: ECMWF==prefix_regex AND DPIRD==exact_key """
if ecmwf.get("mode") != "prefix_regex":
    raise ValueError("ecmwf_daily_nc mode must be prefix_regex")
re.compile(ecmwf["key_regex"])

if dpird.get("mode") != "exact_key":
    raise ValueError("dpird_final_singleton mode must be exact_key")

out = kp["output"]
for k in ["staging_volume_path", "ledger_path", "temp_path"]:
    if not out.get(k):
        raise ValueError(f"Missing output.{k}")

"""Set max_workers to max thread count of system"""
exec_cfg= kp["execution"]
raw_workers = exec_cfg.get("max_workers", "auto")
max_workers = os.cpu_count() if raw_workers in (None, "auto") else int(raw_workers)

## Get keys

Keys managed on Databricks CLI. [Docs for secret management](https://docs.databricks.com/aws/en/security/secrets/)

In [ ]:
ENDPOINT = kp["s3"]["endpoint_url"]
SCOPE = kp["s3"]["secret_scope"]
BUCKET = kp["s3"]["bucket"]
ACCESS_NAME= kp["s3"]["access_key_secret_name"]
SECRET_NAME= kp["s3"]["secret_key_secret_name"]
REGION_NAME= kp["s3"]["region_name"]

ACCESS_KEY = dbutils.secrets.get(scope=SCOPE, key=ACCESS_NAME)
SECRET_KEY = dbutils.secrets.get(scope=SCOPE, key=SECRET_NAME)

Print top-level directories and objects in weather bucket

In [ ]:
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION_NAME,
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

# Print visible buckets
try:
    print("Buckets visible:", [b["Name"] for b in s3.list_buckets().get("Buckets", [])])
except ClientError as e:
    print("list_buckets not allowed:", e.response.get("Error", {}))

# Equivalent to rclone lsd pawsey0411:weather
resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
prefixes = [p["Prefix"] for p in resp.get("CommonPrefixes", [])]
print("Top-level pseudo-directories:", prefixes)

# Top-level objects sample
root_objects = [o["Key"] for o in resp.get("Contents", [])]
print("Top-level objects sample:", root_objects[:20])